<a href="https://colab.research.google.com/github/cduplan59/CFT_analysis/blob/main/ROSG_Test6d_V2_i3_i4_checkpoint_DSI_transmission_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ROSG Test6d V2 — i=3 → i=4 checkpoint DSI transmission audit

**Mode:** Codex + checkpoint/cache.

This notebook reuses i=3 Test6d outputs when available, computes exact i=4 for decisive scenarios, and compares stability of DSI-modulated spectral transmission under one IFS refinement step.

It does **not** prove spontaneous DSI.

## Imports, configuration, scenarios

In [1]:
# ============================================================
# ROSG Test6d V2 — i=3 -> i=4 checkpoint DSI transmission audit
# ------------------------------------------------------------
# Purpose:
#   - Reuse i=3 outputs from Test6d when available.
#   - Compute exact i=4 spectral transmission for decisive scenarios.
#   - Compare whether an explicitly imposed DSI modulation at conductance-link level
#     remains transmitted after one IFS refinement step.
#
# Important:
#   This does not prove spontaneous DSI in ROSG.
# ============================================================

import os, json, math, shutil, zipfile, hashlib, time
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


@dataclass
class Scenario:
    name: str
    amplitude: float
    omega_mode: str = 'lattice'       # lattice, nonlattice
    phase_mode: str = 'coherent'      # coherent, random
    phase_value: float = 0.0


@dataclass
class Test6dV2Config:
    project_name: str = 'ROSG_Test6d_V2_i3_i4_checkpoint_DSI_transmission_audit'
    input_test6d_zip_name: str = 'ROSG_Test6d_DSI_modulated_conductance_channel_audit_results.zip'
    input_test6d_zip_paths: tuple = (
        '/content/drive/MyDrive/ROSG_exports/ROSG_Test6d_DSI_modulated_conductance_channel_audit.zip',
        '/content/drive/MyDrive/ROSG_exports/ROSG_Test6d_DSI_modulated_conductance_channel_audit_results.zip',
        '/content/ROSG_Test6d_DSI_modulated_conductance_channel_audit_results.zip',
        '/mnt/data/ROSG_Test6d_DSI_modulated_conductance_channel_audit_results.zip',
    )
    cache_level_i: int = 3
    compute_level_i: int = 4
    lambda_lattice: float = 2.0
    Z_min: float = -1.0
    Z_max: float = 5.0
    n_Z_dense: int = 65
    u_min: float = 0.03
    u_max: float = 40.0
    n_u: int = 30
    base_conductance: float = 1.0
    vertical_gain: float = 2.1
    activation_center_Z: float = 1.75
    activation_width_Z: float = 0.75
    baseline_poly_degree: int = 3
    permutation_count: int = 300
    random_seed: int = 20260605
    out_dir: str = '/content/ROSG_Test6d_V2_i3_i4_checkpoint_DSI_transmission_audit'
    use_google_drive: bool = True
    drive_dir: str = '/content/drive/MyDrive/ROSG_exports/ROSG_Test6d_V2_i3_i4_checkpoint_DSI_transmission_audit'
    force_recompute_i3: bool = False
    force_recompute_i4: bool = False

CFG = Test6dV2Config()

SCENARIOS = [
    Scenario('H0_smooth', 0.0, 'lattice', 'coherent', 0.0),
    Scenario('H1_lattice_coherent_A002', 0.02, 'lattice', 'coherent', 0.0),
    Scenario('CTRL_nonlattice_coherent_A002', 0.02, 'nonlattice', 'coherent', 0.0),
    Scenario('CTRL_lattice_randomphase_A002', 0.02, 'lattice', 'random', 0.0),
]


def in_colab():
    try:
        import google.colab  # type: ignore
        return True
    except Exception:
        return False


def prepare_output_dir(cfg=CFG):
    out = Path(cfg.out_dir)
    if cfg.use_google_drive and in_colab():
        try:
            from google.colab import drive  # type: ignore
            drive.mount('/content/drive', force_remount=False)
            if Path('/content/drive/MyDrive').exists():
                out = Path(cfg.drive_dir)
        except Exception as exc:
            print('[Drive warning]', exc)
            out = Path(cfg.out_dir)
    out.mkdir(parents=True, exist_ok=True)
    return out

OUT_DIR = prepare_output_dir(CFG)
CACHE_DIR = OUT_DIR / '_cache'
CACHE_DIR.mkdir(parents=True, exist_ok=True)
INPUT_EXTRACT_DIR = OUT_DIR / '_input_Test6d'
INPUT_EXTRACT_DIR.mkdir(parents=True, exist_ok=True)
print('Output directory:', OUT_DIR)

Mounted at /content/drive
Output directory: /content/drive/MyDrive/ROSG_exports/ROSG_Test6d_V2_i3_i4_checkpoint_DSI_transmission_audit


## Input/cache helpers

In [2]:
# ============================================================
# Input/cache helpers
# ============================================================

def resolve_input_test6d_zip(cfg=CFG):
    candidates = [Path(p) for p in cfg.input_test6d_zip_paths]
    candidates += [Path.cwd() / cfg.input_test6d_zip_name, Path('/content') / cfg.input_test6d_zip_name]
    for p in candidates:
        if p.exists() and p.stat().st_size > 0:
            print('[input] found Test6d ZIP:', p)
            return p
    print('[input] no Test6d ZIP found; i=3 will be recomputed if needed.')
    return None


def extract_zip(input_zip, extract_dir=INPUT_EXTRACT_DIR):
    if input_zip is None:
        return None
    if extract_dir.exists():
        shutil.rmtree(extract_dir)
    extract_dir.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(input_zip, 'r') as z:
        z.extractall(extract_dir)
    return extract_dir


def find_file(root, filename):
    if root is None:
        return None
    matches = list(Path(root).rglob(filename))
    return matches[0] if matches else None


def try_load_i3_from_test6d_zip(cfg=CFG):
    if cfg.force_recompute_i3:
        return None
    input_zip = resolve_input_test6d_zip(cfg)
    root = extract_zip(input_zip) if input_zip else None
    if root is None:
        return None
    summary_path = find_file(root, 'test6d_spectral_summary_by_Z.csv')
    if summary_path is None:
        return None
    summary = pd.read_csv(summary_path)
    decisive = [s.name for s in SCENARIOS]
    summary_i3 = summary[(summary['i'] == cfg.cache_level_i) & (summary['scenario'].isin(decisive))].copy()
    if len(summary_i3) == 0:
        return None
    print('[cache] loaded i=3 from previous Test6d ZIP:', len(summary_i3), 'rows')
    return summary_i3

## Graph / conductance / heat trace reconstruction

In [3]:
# ============================================================
# Graph / conductance / heat trace reconstruction
# ============================================================

def omega0_lattice(cfg=CFG):
    return float(2.0 * math.pi / math.log(cfg.lambda_lattice))


def omega_for_scenario(scenario, cfg=CFG):
    omega0 = omega0_lattice(cfg)
    if scenario.omega_mode == 'lattice':
        return omega0
    if scenario.omega_mode == 'nonlattice':
        return float(omega0 * math.sqrt(2.0))
    raise ValueError(f'Unknown omega_mode: {scenario.omega_mode}')


def grid_size_from_i(i):
    return 2 ** int(i)


def num_points_i(i):
    L = grid_size_from_i(i)
    return int((L + 1) ** 2)


def build_Ei(i):
    L = grid_size_from_i(i)
    rows = []
    eid = 0
    def node_id(x, y):
        return y * (L + 1) + x
    for y in range(L + 1):
        for x in range(L + 1):
            u = node_id(x, y)
            if x < L:
                rows.append({'edge_id': eid, 'i': int(i), 'u': int(u), 'v': int(node_id(x+1, y)), 'orientation': 'horizontal'})
                eid += 1
            if y < L:
                rows.append({'edge_id': eid, 'i': int(i), 'u': int(u), 'v': int(node_id(x, y+1)), 'orientation': 'vertical'})
                eid += 1
    return pd.DataFrame(rows)


def local_hierarchy_level_for_edge(row, i):
    L = grid_size_from_i(i)
    u, v = int(row['u']), int(row['v'])
    ux, uy = u % (L + 1), u // (L + 1)
    vx, vy = v % (L + 1), v // (L + 1)
    mx2 = ux + vx
    my2 = uy + vy
    def trailing_zeros(a):
        a = int(abs(a))
        if a == 0:
            return int(i)
        c = 0
        while a % 2 == 0 and c < i:
            c += 1
            a //= 2
        return c
    return int(min(i, max(trailing_zeros(mx2), trailing_zeros(my2))))


def prepare_edge_template(i, cfg=CFG):
    Ei = build_Ei(i)
    hs, thetas = [], []
    for _, row in Ei.iterrows():
        h = local_hierarchy_level_for_edge(row, i)
        theta = cfg.activation_center_Z - 0.12 * h
        hs.append(h)
        thetas.append(theta)
    Ei['hierarchy_h'] = hs
    Ei['theta_edge'] = thetas
    return Ei


def edge_phases(Ei, scenario, i, cfg=CFG):
    if scenario.phase_mode == 'coherent':
        return np.zeros(len(Ei), dtype=float) + float(scenario.phase_value)
    if scenario.phase_mode == 'random':
        rng = np.random.default_rng(cfg.random_seed + 7919 * int(i) + len(scenario.name))
        return rng.uniform(0.0, 2.0 * math.pi, size=len(Ei))
    raise ValueError(f'Unknown phase_mode: {scenario.phase_mode}')


def conductance_edges(Ei, Z, scenario, i, cfg=CFG):
    theta = Ei['theta_edge'].to_numpy(dtype=float)
    m = 1.0 / (1.0 + np.exp(-(float(Z) - theta) / max(cfg.activation_width_Z, 1e-12)))
    c_smooth = cfg.base_conductance + cfg.vertical_gain * m
    omega = omega_for_scenario(scenario, cfg)
    phases = edge_phases(Ei, scenario, i, cfg)
    if scenario.amplitude == 0:
        mod = np.ones_like(c_smooth)
    else:
        mod = 1.0 + float(scenario.amplitude) * np.cos(omega * float(Z) + phases)
    c_eff = np.clip(c_smooth * mod, 1e-9, None)
    Ec = Ei.copy()
    Ec['Z'] = float(Z)
    Ec['scenario'] = scenario.name
    Ec['scenario_amplitude'] = float(scenario.amplitude)
    Ec['scenario_omega'] = float(omega)
    Ec['scenario_phase_mode'] = scenario.phase_mode
    Ec['m_eff'] = m
    Ec['c_smooth'] = c_smooth
    Ec['modulation_factor'] = mod
    Ec['c_eff'] = c_eff
    return Ec


def weighted_laplacian_dense(n, Ec):
    L = np.zeros((n, n), dtype=float)
    for _, e in Ec.iterrows():
        u, v, c = int(e['u']), int(e['v']), float(e['c_eff'])
        L[u, u] += c
        L[v, v] += c
        L[u, v] -= c
        L[v, u] -= c
    return L


def heat_trace_from_eigs(evals, u_grid):
    evals = np.asarray(evals, dtype=float)
    return np.clip(np.array([np.mean(np.exp(-u * np.clip(evals, 0, None))) for u in u_grid], dtype=float), 1e-300, 1.0)


def ds_curve_from_heat(u_grid, P):
    logu = np.log(u_grid)
    logP = np.log(np.clip(P, 1e-300, None))
    return -2.0 * np.gradient(logP, logu)


def ds_eff_from_curve(u_grid, P, ds_curve, n_points):
    p_low = max(5.0 / max(n_points, 1), 1e-4)
    mask = (P < 0.90) & (P > p_low) & np.isfinite(ds_curve) & (ds_curve > -1.0) & (ds_curve < 8.0)
    if int(mask.sum()) < 5:
        lo, hi = np.quantile(np.arange(len(u_grid)), [0.25, 0.75]).astype(int)
        mask = np.zeros_like(u_grid, dtype=bool)
        mask[lo:hi+1] = True
    vals = ds_curve[mask]
    return {
        'ds_eff_median': float(np.median(vals)),
        'ds_eff_mean': float(np.mean(vals)),
        'ds_eff_std': float(np.std(vals)),
        'window_points': int(mask.sum()),
        'u_window_min': float(u_grid[mask].min()),
        'u_window_max': float(u_grid[mask].max()),
    }


def level_summary_path(i):
    return CACHE_DIR / f'level_i_{int(i)}_spectral_summary.csv'


def compute_level_spectral_summary(i, cfg=CFG, scenarios=SCENARIOS):
    if i == cfg.compute_level_i and level_summary_path(i).exists() and not cfg.force_recompute_i4:
        print(f'[cache] load computed level_i={i}')
        return pd.read_csv(level_summary_path(i))
    print(f'[compute] exact spectral scan level_i={i}')
    Z_grid = np.linspace(cfg.Z_min, cfg.Z_max, cfg.n_Z_dense)
    u_grid = np.logspace(math.log10(cfg.u_min), math.log10(cfg.u_max), cfg.n_u)
    mid_idx = len(u_grid)//2
    u_mid = float(u_grid[mid_idx])
    n = num_points_i(i)
    Ei = prepare_edge_template(i, cfg)
    rows = []
    for scenario in scenarios:
        print('  scenario:', scenario.name)
        for Z in Z_grid:
            Ec = conductance_edges(Ei, Z, scenario, i, cfg)
            L = weighted_laplacian_dense(n, Ec)
            evals = np.linalg.eigvalsh(L)
            P = heat_trace_from_eigs(evals, u_grid)
            ds_curve = ds_curve_from_heat(u_grid, P)
            eff = ds_eff_from_curve(u_grid, P, ds_curve, n)
            lambda1 = float(np.sort(evals)[1]) if len(evals) > 1 else float('nan')
            rows.append({
                'i': int(i), 'scenario': scenario.name,
                'amplitude': float(scenario.amplitude), 'omega_mode': scenario.omega_mode, 'phase_mode': scenario.phase_mode,
                'Z': float(Z), 'n_points': int(n), 'num_edges': int(len(Ei)),
                'm_eff_mean': float(Ec['m_eff'].mean()), 'c_smooth_mean': float(Ec['c_smooth'].mean()),
                'modulation_factor_mean': float(Ec['modulation_factor'].mean()), 'modulation_factor_std': float(Ec['modulation_factor'].std(ddof=0)),
                'c_eff_mean': float(Ec['c_eff'].mean()), 'c_eff_std': float(Ec['c_eff'].std(ddof=0)),
                'lambda1': lambda1, 'lambda1_scaled_4powi': float((4**int(i)) * lambda1),
                'u_mid': u_mid, 'P_mid': float(P[mid_idx]), 'logP_mid': float(np.log(P[mid_idx])),
                **eff,
            })
    df = pd.DataFrame(rows)
    df.to_csv(level_summary_path(i), index=False)
    return df

## DSI residual analysis

In [4]:
# ============================================================
# DSI residual analysis
# ============================================================

def fit_smooth_baseline(Z, y, degree=3):
    Z = np.asarray(Z, dtype=float)
    y = np.asarray(y, dtype=float)
    deg = min(int(degree), max(1, len(Z)-2))
    Zc = Z - np.mean(Z)
    coeff = np.polyfit(Zc, y, deg)
    yhat = np.polyval(coeff, Zc)
    rss = float(np.sum((y-yhat)**2))
    return yhat, {'baseline_model': f'poly{deg}', 'baseline_params': [float(x) for x in coeff], 'baseline_rss': rss}


def fit_fixed_frequency(Z, residual, omega):
    Z = np.asarray(Z, dtype=float)
    r = np.asarray(residual, dtype=float)
    X = np.column_stack([np.cos(omega*Z), np.sin(omega*Z), np.ones_like(Z)])
    beta, *_ = np.linalg.lstsq(X, r, rcond=None)
    pred = X @ beta
    rss = float(np.sum((r-pred)**2))
    amp = float(math.sqrt(beta[0]**2 + beta[1]**2))
    phase = float(math.atan2(-beta[1], beta[0]))
    return {'omega': float(omega), 'amplitude': amp, 'phase': phase, 'rss': rss, 'params': [float(x) for x in beta], 'prediction': pred}


def aic_bic(rss, n, k):
    rss = max(float(rss), 1e-300)
    return float(n*math.log(rss/n)+2*k), float(n*math.log(rss/n)+k*math.log(n))


def permutation_pvalue(Z, residual, omega, observed_amp, n_perm, seed):
    rng = np.random.default_rng(seed)
    r = np.asarray(residual, dtype=float)
    amps = []
    for _ in range(int(n_perm)):
        rp = rng.permutation(r)
        amps.append(fit_fixed_frequency(Z, rp, omega)['amplitude'])
    amps = np.asarray(amps)
    return float((np.sum(amps >= observed_amp)+1)/(len(amps)+1)), float(np.mean(amps)), float(np.std(amps))


def frequency_scan(Z, residual, omega_min, omega_max, n_grid=500):
    rows = []
    if omega_max <= omega_min:
        return pd.DataFrame()
    for omega in np.linspace(omega_min, omega_max, n_grid):
        fit = fit_fixed_frequency(Z, residual, float(omega))
        rows.append({k:v for k,v in fit.items() if k != 'prediction'})
    return pd.DataFrame(rows).sort_values('rss').reset_index(drop=True)


def analyze_observable(summary, i, scenario, observable, cfg=CFG):
    omega0 = omega0_lattice(cfg)
    sub = summary[(summary['i'] == i) & (summary['scenario'] == scenario)].sort_values('Z')
    Z = sub['Z'].to_numpy(dtype=float)
    y = sub[observable].to_numpy(dtype=float)
    dZ = float(np.median(np.diff(Z)))
    omega_nyquist = float(math.pi/dZ)
    resolvable = bool(omega0 < 0.8*omega_nyquist)
    baseline, base_info = fit_smooth_baseline(Z, y, cfg.baseline_poly_degree)
    residual = y - baseline
    residual -= np.mean(residual)
    fixed = fit_fixed_frequency(Z, residual, omega0)
    rss_null = float(np.sum(residual**2))
    aic_null, bic_null = aic_bic(rss_null, len(Z), 1)
    aic_fixed, bic_fixed = aic_bic(fixed['rss'], len(Z), 3)
    p_perm, null_amp_mean, null_amp_std = permutation_pvalue(
        Z, residual, omega0, fixed['amplitude'], cfg.permutation_count,
        cfg.random_seed + 1000*i + len(scenario) + len(observable)
    )
    scan_max = min(40.0, 0.95*omega_nyquist)
    scan = frequency_scan(Z, residual, 0.5, scan_max, n_grid=500)
    best_omega = float(scan.iloc[0]['omega']) if len(scan) else float('nan')
    best_rss = float(scan.iloc[0]['rss']) if len(scan) else float('nan')
    omega0_rank = int(np.argmin(np.abs(scan['omega'].to_numpy() - omega0))+1) if len(scan) else None
    if not resolvable:
        verdict = 'not_resolvable'
    elif (aic_null - aic_fixed > 4.0) and (p_perm < 0.05):
        verdict = 'candidate'
    else:
        verdict = 'not_detected'
    row = {
        'i': int(i), 'scenario': scenario, 'observable': observable,
        'n_Z': int(len(Z)), 'dZ_median': dZ,
        'omega0_lattice': omega0, 'omega_nyquist': omega_nyquist, 'omega0_resolvable': resolvable,
        'baseline_model': base_info['baseline_model'], 'baseline_rss': float(base_info['baseline_rss']),
        'residual_rss_null': rss_null,
        'fixed_omega_amplitude': fixed['amplitude'], 'fixed_omega_phase': fixed['phase'], 'fixed_omega_rss': fixed['rss'],
        'delta_aic_null_minus_fixed': float(aic_null - aic_fixed),
        'delta_bic_null_minus_fixed': float(bic_null - bic_fixed),
        'permutation_p_value': p_perm,
        'permutation_amp_mean': null_amp_mean, 'permutation_amp_std': null_amp_std,
        'best_scan_omega': best_omega, 'best_scan_rss': best_rss,
        'omega_distance_to_lattice': float(abs(best_omega - omega0)) if np.isfinite(best_omega) else float('nan'),
        'omega0_rank_nearest': omega0_rank,
        'curve_verdict': verdict,
    }
    curve = pd.DataFrame({'i': int(i), 'scenario': scenario, 'observable': observable, 'Z': Z, 'y': y, 'baseline': baseline, 'residual': residual, 'fixed_omega_fit': fixed['prediction']})
    scan['i'] = int(i); scan['scenario'] = scenario; scan['observable'] = observable
    return row, curve, scan


def run_dsi_analysis(summary, cfg=CFG):
    observables = ['c_eff_mean', 'modulation_factor_mean', 'ds_eff_median', 'ds_eff_mean', 'logP_mid', 'lambda1_scaled_4powi']
    rows, curves, scans = [], [], []
    for i in sorted(summary['i'].unique()):
        for scenario in sorted(summary['scenario'].unique()):
            for obs in observables:
                row, curve, scan = analyze_observable(summary, int(i), scenario, obs, cfg)
                rows.append(row); curves.append(curve); scans.append(scan)
    results = pd.DataFrame(rows)
    curves_df = pd.concat(curves, ignore_index=True)
    scans_df = pd.concat(scans, ignore_index=True)
    results.to_csv(OUT_DIR / 'test6d_v2_DSI_results_by_curve.csv', index=False)
    curves_df.to_csv(OUT_DIR / 'test6d_v2_DSI_residual_curves.csv', index=False)
    scans_df.to_csv(OUT_DIR / 'test6d_v2_DSI_frequency_scan.csv', index=False)

    scenario_summary = []
    for (i, scenario), g in results.groupby(['i', 'scenario']):
        obs_spectral = g[g['observable'].isin(['ds_eff_median', 'ds_eff_mean', 'logP_mid', 'lambda1_scaled_4powi'])]
        candidate_rows = int((g['curve_verdict'] == 'candidate').sum())
        spectral_candidates = int((obs_spectral['curve_verdict'] == 'candidate').sum())
        scenario_summary.append({
            'i': int(i),
            'scenario': scenario,
            'total_rows': int(len(g)),
            'candidate_rows': candidate_rows,
            'spectral_rows': int(len(obs_spectral)),
            'spectral_candidate_rows': spectral_candidates,
            'scenario_verdict': 'transmitted_candidate' if spectral_candidates >= 2 else ('conductance_only_candidate' if candidate_rows > spectral_candidates else 'not_transmitted'),
        })
    scenario_summary = pd.DataFrame(scenario_summary)
    scenario_summary.to_csv(OUT_DIR / 'test6d_v2_scenario_summary_by_i.csv', index=False)

    comparison_rows = []
    for scenario in sorted(scenario_summary['scenario'].unique()):
        g = scenario_summary[scenario_summary['scenario'] == scenario].set_index('i')
        has_i3 = cfg.cache_level_i in g.index
        has_i4 = cfg.compute_level_i in g.index
        verdict_i3 = str(g.loc[cfg.cache_level_i, 'scenario_verdict']) if has_i3 else 'missing'
        verdict_i4 = str(g.loc[cfg.compute_level_i, 'scenario_verdict']) if has_i4 else 'missing'
        stable = bool(verdict_i3 == 'transmitted_candidate' and verdict_i4 == 'transmitted_candidate')
        comparison_rows.append({'scenario': scenario, 'verdict_i3': verdict_i3, 'verdict_i4': verdict_i4, 'stable_transmission_i3_i4': stable})
    comparison = pd.DataFrame(comparison_rows)
    comparison.to_csv(OUT_DIR / 'test6d_v2_i3_i4_transmission_comparison.csv', index=False)

    h1 = comparison[comparison['scenario'] == 'H1_lattice_coherent_A002']
    h0 = comparison[comparison['scenario'] == 'H0_smooth']
    nonlat = comparison[comparison['scenario'] == 'CTRL_nonlattice_coherent_A002']
    randomp = comparison[comparison['scenario'] == 'CTRL_lattice_randomphase_A002']
    h1_stable = bool(len(h1) and h1.iloc[0]['stable_transmission_i3_i4'])
    controls_rejected = bool(
        (len(h0)==0 or not h0.iloc[0]['stable_transmission_i3_i4']) and
        (len(nonlat)==0 or not nonlat.iloc[0]['stable_transmission_i3_i4']) and
        (len(randomp)==0 or not randomp.iloc[0]['stable_transmission_i3_i4'])
    )
    if h1_stable and controls_rejected:
        global_verdict = 'stable_DSI_transmission_i3_i4_controls_rejected'
    elif h1_stable:
        global_verdict = 'stable_DSI_transmission_i3_i4_controls_not_fully_rejected'
    else:
        global_verdict = 'not_stable_i3_i4'

    report = {
        'status': 'completed',
        'not_a_DSI_proof': True,
        'global_verdict': global_verdict,
        'omega0_lattice': omega0_lattice(cfg),
        'cache_level_i': int(cfg.cache_level_i),
        'compute_level_i': int(cfg.compute_level_i),
        'scenarios': [s.name for s in SCENARIOS],
        'scenario_summary_by_i': scenario_summary.to_dict(orient='records'),
        'i3_i4_comparison': comparison.to_dict(orient='records'),
        'interpretation': 'Stable i=3->i=4 transmission means an explicitly imposed DSI modulation survives one IFS refinement through the spectral pipeline; it is not a proof of spontaneous DSI.',
    }
    with open(OUT_DIR / 'test6d_v2_report.json', 'w', encoding='utf-8') as f:
        json.dump(report, f, indent=2, ensure_ascii=False)
    return results, curves_df, scans_df, scenario_summary, comparison, report

## Figures and export

In [5]:
# ============================================================
# Figures and export
# ============================================================

def make_figures(summary, curves_df, scenario_summary, comparison, cfg=CFG):
    fig_dir = OUT_DIR / 'figures'
    fig_dir.mkdir(parents=True, exist_ok=True)
    for obs in ['ds_eff_median', 'logP_mid', 'lambda1_scaled_4powi']:
        for i in sorted(summary['i'].unique()):
            plt.figure(figsize=(9,5))
            sub = summary[summary['i'] == i]
            for scenario, g in sub.groupby('scenario'):
                plt.plot(g['Z'], g[obs], label=scenario)
            plt.xlabel('Z'); plt.ylabel(obs)
            plt.title(f'Test6d V2 observable — {obs}, i={i}')
            plt.grid(True, alpha=0.3); plt.legend(fontsize=8); plt.tight_layout()
            plt.savefig(fig_dir / f'observable_{obs}_i{i}.png', dpi=160)
            plt.close()
    for obs in ['ds_eff_median', 'logP_mid', 'lambda1_scaled_4powi']:
        for i in sorted(curves_df['i'].unique()):
            plt.figure(figsize=(9,5))
            sub = curves_df[(curves_df['i'] == i) & (curves_df['observable'] == obs)]
            for scenario, g in sub.groupby('scenario'):
                plt.plot(g['Z'], g['residual'], label=scenario)
            plt.axhline(0, linewidth=1)
            plt.xlabel('Z'); plt.ylabel('residual')
            plt.title(f'Test6d V2 residual — {obs}, i={i}')
            plt.grid(True, alpha=0.3); plt.legend(fontsize=8); plt.tight_layout()
            plt.savefig(fig_dir / f'residual_{obs}_i{i}.png', dpi=160)
            plt.close()
    plt.figure(figsize=(9,5))
    pivot = scenario_summary.pivot(index='scenario', columns='i', values='spectral_candidate_rows').fillna(0)
    x = np.arange(len(pivot.index)); width = 0.35; levels = list(pivot.columns)
    for k, level in enumerate(levels):
        plt.bar(x + (k - (len(levels)-1)/2)*width, pivot[level], width, label=f'i={level}')
    plt.xticks(x, pivot.index, rotation=45, ha='right', fontsize=8)
    plt.ylabel('spectral candidate rows')
    plt.title('Test6d V2 i=3 -> i=4 spectral transmission')
    plt.legend(); plt.tight_layout()
    plt.savefig(fig_dir / 'scenario_spectral_candidate_rows_i3_i4.png', dpi=160)
    plt.close()


def sha256_file(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(1024*1024), b''):
            h.update(chunk)
    return h.hexdigest()


def write_readme_manifest_and_zip():
    lines = [
        '# ROSG Test6d V2 — i=3 -> i=4 checkpoint DSI transmission audit',
        '',
        'Purpose:',
        '- reuse i=3 Test6d outputs when available;',
        '- compute exact i=4 spectral transmission for decisive scenarios;',
        '- compare H0, H1 lattice coherent A=0.02, non-lattice control, and random-phase control;',
        '- decide whether DSI-modulated conductance transmission is stable under one IFS refinement step.',
        '',
        'Important:',
        'This does not prove spontaneous DSI in ROSG.',
        'It tests whether an explicitly imposed link-level DSI modulation survives from i=3 to i=4 through the heat-trace/spectral pipeline.',
        '',
        'Main files:',
        '- test6d_v2_spectral_summary_i3_i4.csv',
        '- test6d_v2_DSI_results_by_curve.csv',
        '- test6d_v2_scenario_summary_by_i.csv',
        '- test6d_v2_i3_i4_transmission_comparison.csv',
        '- test6d_v2_report.json',
        '- figures/',
        '- manifest_sha256.csv',
    ]
    (OUT_DIR / 'README.md').write_text('\n'.join(lines), encoding='utf-8')
    rows = []
    for p in sorted(OUT_DIR.rglob('*')):
        if p.is_file() and p.name != 'manifest_sha256.csv':
            rows.append({'path': str(p.relative_to(OUT_DIR)), 'size_bytes': int(p.stat().st_size), 'sha256': sha256_file(p)})
    pd.DataFrame(rows).to_csv(OUT_DIR / 'manifest_sha256.csv', index=False)
    zip_path = shutil.make_archive(str(OUT_DIR), 'zip', OUT_DIR)
    print('ZIP:', zip_path)
    return zip_path


def run_autotests():
    print('Running Test6d V2 autotests...')
    omega0 = omega0_lattice(CFG)
    dZ = (CFG.Z_max-CFG.Z_min)/(CFG.n_Z_dense-1)
    assert omega0 < 0.8*(math.pi/dZ)
    assert num_points_i(CFG.compute_level_i) <= 400
    Ei = prepare_edge_template(3, CFG)
    Ec0 = conductance_edges(Ei, 0.0, SCENARIOS[0], 3, CFG)
    Ec1 = conductance_edges(Ei, 0.0, SCENARIOS[1], 3, CFG)
    assert np.all(Ec0['c_eff'].to_numpy() > 0)
    assert np.all(Ec1['c_eff'].to_numpy() > 0)
    print('All autotests passed.')


def run_full_audit(cfg=CFG):
    summary_i3 = try_load_i3_from_test6d_zip(cfg)
    if summary_i3 is None or len(summary_i3) == 0:
        print('[fallback] recomputing i=3')
        summary_i3 = compute_level_spectral_summary(cfg.cache_level_i, cfg, SCENARIOS)
    summary_i4 = compute_level_spectral_summary(cfg.compute_level_i, cfg, SCENARIOS)
    summary = pd.concat([summary_i3, summary_i4], ignore_index=True)
    summary.to_csv(OUT_DIR / 'test6d_v2_spectral_summary_i3_i4.csv', index=False)
    results, curves_df, scans_df, scenario_summary, comparison, report = run_dsi_analysis(summary, cfg)
    make_figures(summary, curves_df, scenario_summary, comparison, cfg)
    zip_path = write_readme_manifest_and_zip()
    return summary, results, curves_df, scans_df, scenario_summary, comparison, report, zip_path

## Run Test6d V2

In [6]:
run_autotests()
summary, results, curves_df, scans_df, scenario_summary, comparison, report, zip_path = run_full_audit(CFG)
print(json.dumps({
    "status": "completed",
    "zip_path": zip_path,
    "global_verdict": report["global_verdict"],
    "omega0_lattice": report["omega0_lattice"],
    "cache_level_i": report["cache_level_i"],
    "compute_level_i": report["compute_level_i"],
}, indent=2))
display(comparison)
display(scenario_summary)


Running Test6d V2 autotests...
All autotests passed.
[input] no Test6d ZIP found; i=3 will be recomputed if needed.
[fallback] recomputing i=3
[compute] exact spectral scan level_i=3
  scenario: H0_smooth
  scenario: H1_lattice_coherent_A002
  scenario: CTRL_nonlattice_coherent_A002
  scenario: CTRL_lattice_randomphase_A002
[cache] load computed level_i=4
ZIP: /content/drive/MyDrive/ROSG_exports/ROSG_Test6d_V2_i3_i4_checkpoint_DSI_transmission_audit.zip
{
  "status": "completed",
  "zip_path": "/content/drive/MyDrive/ROSG_exports/ROSG_Test6d_V2_i3_i4_checkpoint_DSI_transmission_audit.zip",
  "global_verdict": "stable_DSI_transmission_i3_i4_controls_rejected",
  "omega0_lattice": 9.064720283654388,
  "cache_level_i": 3,
  "compute_level_i": 4
}


,scenario,verdict_i3,verdict_i4,stable_transmission_i3_i4
0,CTRL_lattice_randomphase_A002,conductance_only_candidate,conductance_only_candidate,False
1,CTRL_nonlattice_coherent_A002,not_transmitted,not_transmitted,False
2,H0_smooth,not_transmitted,not_transmitted,False
3,H1_lattice_coherent_A002,transmitted_candidate,transmitted_candidate,True


,i,scenario,total_rows,candidate_rows,spectral_rows,spectral_candidate_rows,scenario_verdict
0,3,CTRL_lattice_randomphase_A002,6,1,4,0,conductance_only_candidate
1,3,CTRL_nonlattice_coherent_A002,6,0,4,0,not_transmitted
2,3,H0_smooth,6,0,4,0,not_transmitted
3,3,H1_lattice_coherent_A002,6,4,4,2,transmitted_candidate
4,4,CTRL_lattice_randomphase_A002,6,1,4,0,conductance_only_candidate
5,4,CTRL_nonlattice_coherent_A002,6,0,4,0,not_transmitted
6,4,H0_smooth,6,0,4,0,not_transmitted
7,4,H1_lattice_coherent_A002,6,6,4,4,transmitted_candidate
